In [1]:
# Imports

import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe        # For hypeparameter tuning of deep learning model
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import mlflow
from mlflow.models import infer_signature

In [2]:
# Load dataset

data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)

data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [3]:
# Split data into train, test and validation sets

train,test=train_test_split(data, test_size=0.2, random_state=44)

X_train = train.drop(['quality'], axis=1).values
y_train = train[['quality']].values.ravel()

# Test dataset
X_test = test.drop(['quality'], axis=1).values
y_test = test[['quality']].values.ravel()

# Training and validation data
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train,test_size=0.20, random_state=50)

# Signature
signature = infer_signature(X_train,y_train)

In [4]:
# ANN model trainer function

# Model trainer
def train_model(params,epochs, X_train,y_train,X_val,y_val,X_test,y_test):
    
    # Define model architecture
    mean = np.mean(X_train, axis=0)        # Calculate mean value per independent feature
    var = np.var(X_train,axis=0)           # Calculate variance value per independent feature

    model = keras.Sequential(
        [
            keras.Input([X_train.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dense(1)
        ]
    )

    # Compile model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params['lr'],
        momentum=params['momentum']
    ),
    loss= "mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    # Train model with ML Flow tracking
    with mlflow.start_run(nested=True):
        model.fit(X_train,y_train,epochs=epochs, batch_size=64, validation_data=(X_val,y_val))

        # Evaluate model
        eval_result = model.evaluate(X_val,y_val, batch_size=64)

        eval_rsme = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rsme)

        # Log model
        mlflow.tensorflow.log_model(model, "ANN model", signature=signature)

        return {"loss": eval_rsme, "status" : STATUS_OK, "model" : model}

In [5]:
# Objective function

def objective(params):

    # ML Flow will track the parameters and results for each run

    result = train_model(
        params,
        epochs=3,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test
    )
    return result

parameter_space = {
    'lr' : hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum" : hp.uniform("momentum", 0.0, 1.0)
}

In [10]:
# Hyperparameter Tuning

mlflow.set_experiment("Wine Quality")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

with mlflow.start_run():
    # Hyperparameter search
    trials = Trials()
    best = fmin(
        fn= objective,
        space=parameter_space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Details of best run
    best_run = sorted(trials.results, key=lambda x:x['loss'])[0]

    # Log best parameters
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run['loss'])
    mlflow.tensorflow.log_model(best_run['model'], "model", signature=signature)

    # Print best parameters
    print(f"Best Parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 17s 367ms/step - loss: 34.2314 - root_mean_squared_error: 5.8508
36/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 10.4084 - root_mean_squared_error: 3.0700   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.6888 - root_mean_squared_error: 1.9206 - val_loss: 1.2309 - val_root_mean_squared_error: 1.1095

Epoch 2/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 1.0751 - root_mean_squared_error: 1.0369
37/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0561 - root_mean_squared_error: 1.0267 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9077 - root_mean_squared_error: 0.9527 - val_loss: 0.7649 - val_root_mean_squared_error: 0.8746

Epoch 3/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - loss: 0.9702 - root_mean_squared_error: 0.9850
23/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6973 - root_mean_squared_error: 0.8343 
49/

2026/04/24 22:02:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run rumbling-bug-25 at: http://127.0.0.1:5000/#/experiments/1/runs/ac85428d67d34f36afe17d1443f95ac2

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 15s 326ms/step - loss: 36.7919 - root_mean_squared_error: 6.0656
42/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 35.2009 - root_mean_squared_error: 5.9330   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 35.2274 - root_mean_squared_error: 5.9353 - val_loss: 34.7711 - val_root_mean_squared_error: 5.8967

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 36.6900 - root_mean_squared_error: 6.0572
37/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 34.4388 - root_mean_squared_error: 5.8683 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 34.0382 - root_mean_squared_error: 5.8342 - val_loss: 33.5984 - val_root_mean_squared_error: 5.7964

Epoch 3/3

2026/04/24 22:02:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run agreeable-fly-510 at: http://127.0.0.1:5000/#/experiments/1/runs/38c8fe60be794544a961476748690bb6

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 17s 363ms/step - loss: 35.4302 - root_mean_squared_error: 5.9523
43/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 27.6036 - root_mean_squared_error: 5.2428   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 20.9956 - root_mean_squared_error: 4.5821 - val_loss: 11.2304 - val_root_mean_squared_error: 3.3512

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 9.9436 - root_mean_squared_error: 3.1533
44/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 8.6619 - root_mean_squared_error: 2.9373 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.6933 - root_mean_squared_error: 2.5871 - val_loss: 3.8735 - val_root_mean_squared_error: 

...

🏃 View run resilient-doe-74 at: http://127.0.0.1:5000/#/experiments/1/runs/b8b258f1c0cd44b2a8a0aa4f19791f24

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 18s 393ms/step - loss: 37.6885 - root_mean_squared_error: 6.1391
35/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 33.4737 - root_mean_squared_error: 5.7817   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 26.7326 - root_mean_squared_error: 5.1704 - val_loss: 17.4414 - val_root_mean_squared_error: 4.1763

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - loss: 18.1572 - root_mean_squared_error: 4.2611
38/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 14.9545 - root_mean_squared_error: 3.8624 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 11.7353 - root_mean_squared_error: 3.4257 - val_loss: 7.5048 - val_root_mean_squared_error

2026/04/24 22:03:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run classy-colt-383 at: http://127.0.0.1:5000/#/experiments/1/runs/71577657131c44eba0f3e23d81d4a584

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

100%|██████████| 4/4 [00:36<00:00,  9.03s/trial, best loss: 0.7877904176712036]

2026/04/24 22:03:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best Parameters: {'lr': np.float64(0.009787279492805303), 'momentum': np.float64(0.623832641998661)}
Best eval rmse: 0.7877904176712036
🏃 View run angry-duck-451 at: http://127.0.0.1:5000/#/experiments/1/runs/4e8d651b979c46138e732fdd09371714
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
